# Laubmann-KG — Book 2 (Vol. 2)

Runs the KG pipeline over the **whole of Book 2** with the **Gemini** extraction
backend, compares it against the offline **rule-based** backend, and explores /
visualizes the resulting knowledge graph.

Input is `entries.csv` (all volumes) filtered to `sample.volume: 2`. The
`by_volume/Laubmann_02.md` file is the human-readable rendering, not a pipeline
input, so it is not used here.

All cells assume the working directory is the repo root (`/content/laubmann-kg_TP`),
which Cell 1 sets.

## 0 · Setup

In [ ]:
import os
if not os.path.isdir('/content/laubmann-kg_TP'):
    !git clone https://github.com/Maelkolb/laubmann-kg_TP.git
%cd /content/laubmann-kg_TP
!pip -q install -e ".[llm]"
!pip -q install pyvis networkx

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CORPUS    = '/content/drive/MyDrive/HistOrniGraph_output/corpus_2026-07-21'
MODEL     = 'gemini-3.5-flash'      # matches configs/models.yaml; change if needed
VALIDATE  = True                   # True = abort export on SHACL violations

OUT_LLM   = 'data/exports_vol2_llm'
OUT_RULES = 'data/exports_vol2_rules'

In [ ]:
import os
from google.colab import userdata

key = userdata.get('LST_Gemini')          # your Colab secret name
os.environ['GOOGLE_API_KEY'] = key         # client reads this first,
os.environ['GEMINI_API_KEY'] = key         # falls back to this
print('Gemini key loaded:', bool(key))

In [ ]:
import yaml, pandas as pd

cfg = yaml.safe_load(open('configs/sample_llm.yaml'))
cfg['extraction']['model'] = MODEL
yaml.safe_dump(cfg, open('configs/sample_llm.yaml', 'w'),
               sort_keys=False, allow_unicode=True)
print('LLM extraction model:', cfg['extraction']['model'])

_df   = pd.read_csv(f'{CORPUS}/entries.csv')
_vol2 = _df[_df['volume'].astype(str).str.strip() == '2']
print(f'Vol. 2 entries: {len(_vol2)}   (corpus total: {len(_df)})')
assert len(_vol2) > 0, 'No volume-2 rows — check the volume column values.'

In [ ]:
from rdflib import RDF
from laubmann_kg.kg.sparql import load_graph
from laubmann_kg.kg.rdf import LKG

OUT_LLM = 'data/exports_vol2_llm'
ttl, jsonld = f'{OUT_LLM}/rdf/laubmann_sample.ttl', f'{OUT_LLM}/jsonld/laubmann_sample.jsonld'
g = load_graph(ttl)
summary_llm = {
      'ttl': ttl, 'jsonld': jsonld, 'triples': len(_g),
      'entries':      len(set(_g.subjects(RDF.type, LKG.DiaryEntry))),
      'observations': len(set(_g.subjects(RDF.type, LKG.Observation))),
      'shacl_conforms': False,   # this export predates da5739f
  }
print(summary_llm)

## 1 · Full Book 2 run — Gemini backend

One Gemini call per entry, forced to JSON at temperature 0. Responses are cached
under `data/cache/llm/`, so re-running this cell is free and deterministic.

The returned summary carries entry / observation / triple counts and the paths of
the written TTL + JSON-LD.

In [ ]:
import sys
import os

# Add the 'src' directory to the Python path to ensure 'laubmann_kg' is found
# Assuming the current working directory is /content/laubmann-kg_TP as set in Cell 1
repo_root = os.getcwd()
src_path = os.path.join(repo_root, 'src')
if src_path not in sys.path:
    sys.path.append(src_path)

from laubmann_kg.pipeline import load_config
from laubmann_kg.kg import export

summary_llm = export(load_config('configs/sample_llm.yaml'), CORPUS, OUT_LLM,
                     validate=VALIDATE)
summary_llm

In [ ]:
# Optional: Darwin Core Archive for the same run
from laubmann_kg.dwca import export as export_dwca
export_dwca(load_config('configs/sample_llm.yaml'), CORPUS, OUT_LLM, validate=VALIDATE)

## 2 · Compare with the heuristic (rule-based) backend

Same Book 2 entries, offline gazetteer/rule extraction — no API calls. The
overview compares totals; the species table shows what each backend surfaced via
the `CQ1_species_frequency` competency question.

In [ ]:
from laubmann_kg.kg import export
from laubmann_kg.kg.sparql import load_graph, run_query
import pandas as pd

summary_rules = export(load_config('configs/sample.yaml'), CORPUS, OUT_RULES,
                       validate=VALIDATE)

cols = ('entries', 'observations', 'triples', 'shacl_conforms')
overview = pd.DataFrame([
    {'backend': 'gemini', **{k: summary_llm[k]   for k in cols}},
    {'backend': 'rules',  **{k: summary_rules[k] for k in cols}},
]).set_index('backend')
print(overview, '\n')

def top_species(ttl, n=15):
    rows = run_query(load_graph(ttl), 'CQ1_species_frequency')[:n]
    return pd.DataFrame(rows, columns=['vernacular', 'n']).rename(columns={'n': 'count'})

llm_sp   = top_species(summary_llm['ttl']).rename(columns={'count': 'gemini'})
rules_sp = top_species(summary_rules['ttl']).rename(columns={'count': 'rules'})
compare  = pd.merge(llm_sp, rules_sp, on='vernacular', how='outer').fillna(0)
compare[['gemini', 'rules']] = compare[['gemini', 'rules']].astype(int)
compare.sort_values('gemini', ascending=False, ignore_index=True)

In [ ]:
  from rdflib import RDF
  from laubmann_kg.kg.sparql import load_graph
  from laubmann_kg.kg.rdf import LKG

  OUT_LLM = 'data/exports_vol2_llm'
  ttl, jsonld = f'{OUT_LLM}/rdf/laubmann_sample.ttl', f'{OUT_LLM}/jsonld/laubmann_sample.jsonld'
  _g = load_graph(ttl)
  summary_llm = {
      'ttl': ttl, 'jsonld': jsonld, 'triples': len(_g),
      'entries':      len(set(_g.subjects(RDF.type, LKG.DiaryEntry))),
      'observations': len(set(_g.subjects(RDF.type, LKG.Observation))),
      'shacl_conforms': False,   # this export predates da5739f
  }
  print(summary_llm)

## 3 · Explore the graph — competency questions

Runs all six SPARQL competency questions defined in `kg/sparql.py` against the
Gemini graph and shows the first rows of each.

In [ ]:
from laubmann_kg.kg.sparql import load_graph, run_all, QUERIES
import pandas as pd

g = load_graph(summary_llm['ttl'])
print('Triples:', len(g))
print('Competency questions:', list(QUERIES), '\n')

for cq, rows in run_all(g).items():
    print(f'=== {cq}  ({len(rows)} rows) ===')
    display(pd.DataFrame(rows).head(15))
    print()

## 4 · Interactive graph visualization

An observation-centered subgraph (each `Observation` and its neighborhood),
colored by node type. Adjust `MAX_OBS` / `MAX_EDGES` to widen or thin it. Falls
back to a static Matplotlib layout if pyvis can't render.

In [ ]:
import html
from rdflib import RDF, RDFS, Namespace, Literal
from laubmann_kg.kg.sparql import load_graph

LKG = Namespace('https://w3id.org/laubmann-kg/ontology#')
DWC = Namespace('http://rs.tdwg.org/dwc/terms/')
g   = load_graph(summary_llm['ttl'])

MAX_OBS   = 60
MAX_EDGES = 800

TYPE_COLORS = {
    'Observation': '#e4572e', 'Taxon': '#2e86ab', 'DiaryEntry': '#8a4fff',
    'Place': '#3bb273', 'Vocalisation': '#f3a712',
}
URI_COLOR, LIT_COLOR = '#8899aa', '#c9d6df'

def local(t):
    s = str(t)
    return s.rsplit('#', 1)[-1].rsplit('/', 1)[-1]

def node_type(n):
    for t in g.objects(n, RDF.type):
        if local(t) in TYPE_COLORS:
            return local(t)
    return None

def label_for(n):
    if isinstance(n, Literal):
        s = str(n)
        return s[:40] + '…' if len(s) > 40 else s
    for pred in (DWC.vernacularName, RDFS.label):
        for o in g.objects(n, pred):
            return str(o)
    return local(n)

obs = list(g.subjects(RDF.type, LKG.Observation))[:MAX_OBS]
edges, seen, frontier = [], set(), list(obs)
for _ in range(2):
    nxt = []
    for s in frontier:
        for p, o in g.predicate_objects(s):
            if p == RDF.type or (isinstance(o, Literal) and len(str(o)) > 60):
                continue
            edges.append((s, p, o))
            if not isinstance(o, Literal) and o not in seen:
                seen.add(o); nxt.append(o)
            if len(edges) >= MAX_EDGES:
                break
        if len(edges) >= MAX_EDGES:
            break
    frontier = nxt

print(f'{len(obs)} observation events → {len(edges)} edges')

In [ ]:
def render_pyvis(edges, path='kg_vol2.html'):
    from pyvis.network import Network
    net = Network(height='650px', width='100%', bgcolor='#ffffff', font_color='#222',
                  notebook=False, cdn_resources='remote', directed=True)
    net.barnes_hut(gravity=-8000, spring_length=120)
    added = set()
    def add(n):
        nid = str(n)
        if nid not in added:
            if isinstance(n, Literal):
                color, shape = LIT_COLOR, 'box'
            else:
                color, shape = TYPE_COLORS.get(node_type(n), URI_COLOR), 'dot'
            net.add_node(nid, label=label_for(n), color=color, shape=shape,
                         title=html.escape(str(n)))
            added.add(nid)
        return nid
    for s, p, o in edges:
        net.add_edge(add(s), add(o), label=local(p), title=local(p))
    net.save_graph(path)
    return path

from IPython.display import HTML, display
try:
    display(HTML(filename=render_pyvis(edges)))
except Exception as exc:
    print('pyvis fallback:', exc)
    import networkx as nx, matplotlib.pyplot as plt
    G = nx.DiGraph()
    for s, p, o in edges[:250]:
        G.add_edge(label_for(s), label_for(o))
    plt.figure(figsize=(14, 10))
    nx.draw(G, node_size=250, font_size=7, with_labels=True,
            node_color='#2e86ab', edge_color='#cccccc')
    plt.show()

## 5 · Download

In [ ]:
from google.colab import files
files.download('kg_vol2.html')
# Bundle the full exports:
# !zip -qr exports_vol2.zip data/exports_vol2_llm data/exports_vol2_rules
# files.download('exports_vol2.zip')

In [ ]:
from google.colab import files
!zip -qr exports_vol2.zip data/exports_vol2_llm data/exports_vol2_rules
files.download('exports_vol2.zip')